# Anarkali Web — train on real web pages

One notebook, start to finish, on Colab or Kaggle (GPU runtime: T4 is enough).

1. Real pages are streamed from **Common Crawl**.
2. Labels come free from each page's own **schema.org** markup (the model never sees that markup).
3. The model learns 3 decisions: `page_type`, `price_field` and `in_stock`.
4. It is scored on the test split (pages from sites it never saw) against simple baselines.
5. The model is exported to ONNX, and `best.pt` is saved too, so it is never lost again.

Every cell's output is also written, cell by cell, to `anarkali-web-log.txt` (in `/content` or `/kaggle/working`, and copied to the output folder at the end). If something fails, send that file.

Checkpoints and data are uploaded to a **private** Hugging Face repo as soon as they exist, so a Colab disconnect loses nothing. Add an `HF_TOKEN` secret (write access) or paste the token when asked; it is never saved in the notebook.

**Runtime → Change runtime type → GPU**, then *Run all*.
On Kaggle: Settings → Internet **On**, Accelerator **GPU**.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/ToufiqQureshi/anarkali.git"
BRANCH = "main"

RUN_NAME = "run2"               # results go to HF_REPO/<RUN_NAME>/ and the run log in CLAUDE.md
HF_REPO = "toufiqqureshi651/anarkali-web"   # private repo for checkpoints and data; needs an HF token
INIT_FROM = "run1/web-run/best.pt"           # continue from this checkpoint in HF_REPO; "" = start fresh

BASE_MODEL = "jhu-clsp/ettin-encoder-150m"   # must match the INIT_FROM checkpoint
CRAWL = "CC-MAIN-2026-39"       # fixed crawl so runs are comparable; "latest" for the newest
WARCS = 12                      # each WARC holds ~30-40k HTML pages (~1 GB download)
MAX_PAGES_PER_WARC = 40_000
MAX_PAGES_PER_HOST = 20         # stops a single big site from dominating the data
NON_PRODUCT_KEEP = 0.3          # keep every product page but only 30% of the rest (page_type has plenty)

MIX_OLD_DATA = True             # mix 15% of the old typed-decisions data so the general format stays stable
MIX_RATIO = 0.15

EPOCHS = 3
BATCH_SIZE = 16
MAX_TOKENS = 384                # 512 reads more of each page but trains ~25-30% slower
ENCODER_LR = 3e-5

JULIA_BASELINE = True           # also score SupersonicLabs/Julia-1 on the same test split

## 2. Setup

In [ ]:
import os, subprocess, sys, json, shutil, datetime, traceback
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
REPO = WORK / "anarkali"
LOG = WORK / "anarkali-web-log.txt"   # every cell's output, cell by cell

def _log(text):
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(text)

class _Tee:
    """Send what a cell prints to the notebook and to LOG."""
    def __init__(self, stream):
        self.stream = stream
    def write(self, text):
        self.stream.write(text)
        _log(text)
        return len(text)
    def flush(self):
        self.stream.flush()
    def __getattr__(self, name):
        return getattr(self.stream, name)

def _cell_start(info=None, title=None):
    for name in ("stdout", "stderr"):
        if not isinstance(getattr(sys, name), _Tee):
            setattr(sys, name, _Tee(getattr(sys, name)))
    if title is None:
        title = next((l for l in info.raw_cell.strip().splitlines() if l.strip()), "")[:90]
    _log(f"\n===== {datetime.datetime.now():%Y-%m-%d %H:%M:%S} | {title} =====\n")

def _cell_end(result):
    if result.error_in_exec is not None:
        _log("ERROR:\n" + "".join(traceback.format_exception(result.error_in_exec)))

ip = get_ipython()
for event, fn in (("pre_run_cell", _cell_start), ("post_run_cell", _cell_end)):
    for old in [f for f in ip.events.callbacks[event] if getattr(f, "__name__", "") == fn.__name__]:
        ip.events.unregister(event, old)
    ip.events.register(event, fn)
_cell_start(title="2. Setup")

def sh(cmd):
    """Run a shell command, streaming its output to the notebook and LOG; raise if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=str(REPO) if REPO.exists() else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace",
                            bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode, cmd)

if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
os.chdir(REPO)
sh('pip install -q -e ".[train,onnx]" warcio requests datasets onnx')

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")
print("log file:", LOG)

def _hf_token():
    token = os.environ.get("HF_TOKEN")
    for getter in ("google.colab.userdata", "kaggle_secrets"):
        if token:
            break
        try:
            if getter == "google.colab.userdata":
                from google.colab import userdata
                token = userdata.get("HF_TOKEN")
            else:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face write token (not saved): ").strip()
    return token

from huggingface_hub import HfApi, hf_hub_download
HF_TOKEN = _hf_token()
hf = HfApi(token=HF_TOKEN)
print("Hugging Face user:", hf.whoami()["name"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = REPO / "artifacts" / "web-decisions-v0"
RUN = REPO / "artifacts" / "web-run"
RELEASE = REPO / "artifacts" / "release-web"

## 3. Build the dataset from Common Crawl

It takes a few minutes per WARC. Progress prints every 2,000 pages.

In [ ]:
mix_flag = ""
if MIX_OLD_DATA:
    sh("python scripts/prepare_typed_decisions.py --question-types choice,noul,score --output artifacts/typed-decisions-v2")
    mix_flag = f"--mix artifacts/typed-decisions-v2 --mix-ratio {MIX_RATIO}"
sh(f"python scripts/build_web_decisions.py --crawl {CRAWL} --warcs {WARCS} "
   f"--max-pages-per-warc {MAX_PAGES_PER_WARC} --max-pages-per-host {MAX_PAGES_PER_HOST} "
   f"--non-product-keep {NON_PRODUCT_KEEP} --output {DATA} {mix_flag}")
# Save the dataset now, so a disconnect during training does not lose it.
hf.create_repo(HF_REPO, private=True, exist_ok=True)
hf.upload_folder(repo_id=HF_REPO, folder_path=str(DATA), path_in_repo=f"{RUN_NAME}/web-decisions-v0")
print("dataset uploaded to", f"{HF_REPO}/{RUN_NAME}/web-decisions-v0")

## 4. What the data looks like

In [ ]:
manifest = json.loads((DATA / "manifest.json").read_text())
print("crawl:", manifest.get("crawl"), "| pages:", manifest.get("collection_stats"))
print("old rows mixed into train:", manifest["mixed_train_rows"])
for split, info in manifest["split_counts"].items():
    print(f"\n{split}: {info['decision_cases']} decisions from {info['source_groups']} sites")
    for label, n in info["labels"].items():
        print(f"   {label:30s} {n}")

first = json.loads(next(open(DATA / "train.jsonl")))
print("\nExample state the model reads:")
print(json.dumps(first["state"], ensure_ascii=False)[:600])

## 5. Train

Progress prints live: one line every 25 steps and one per epoch. The best epoch is saved as `best.pt` after each epoch.

In [ ]:
init_flag = ""
if INIT_FROM:
    init_path = hf_hub_download(HF_REPO, INIT_FROM, token=HF_TOKEN)
    init_flag = f"--init-checkpoint {init_path}"
    print("starting from", f"{HF_REPO}/{INIT_FROM}")
sh(f"python scripts/train_anarkali.py --data {DATA} --output {RUN} --architecture packed "
   f"--model {BASE_MODEL} --packed-max-tokens {MAX_TOKENS} --epochs {EPOCHS} --batch-size {BATCH_SIZE} "
   f"--encoder-lr {ENCODER_LR} --selection-metric accuracy_then_ce --schedule linear --warmup-ratio 0.06 "
   f"--log-every 100 --device {DEVICE} {init_flag}")
hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(RUN / "best.pt"), path_in_repo=f"{RUN_NAME}/web-run/best.pt")
print("best.pt uploaded to", f"{HF_REPO}/{RUN_NAME}/web-run/best.pt")

## 6. Test on unseen sites

Temperature is fitted on the calibration split, then the test split is scored once.

In [ ]:
sh(f"python scripts/evaluate_checkpoint.py --checkpoint {RUN}/best.pt --data {DATA} --device {DEVICE} --output {RUN}")
report = json.loads((RUN / "test-report.json").read_text())

In [ ]:
from collections import Counter, defaultdict

def key(case_id):
    return case_id.rsplit("::", 1)[-1]

def load(path):
    return [json.loads(line) for line in open(path) if line.strip()]

train = [r for r in load(DATA / "train.jsonl") if r["workflow"] == "web"]
test_rows = {r["case_id"]: r for r in load(DATA / "test.jsonl")}
preds = load(RUN / "anarkali-test.jsonl")

majority = {}
for q in ("page_type", "in_stock"):
    labels = Counter(r["candidates"][max(range(len(r["target"])), key=r["target"].__getitem__)]["id"]
                     for r in train if key(r["case_id"]) == q)
    majority[q] = labels.most_common(1)[0][0] if labels else None

stats = defaultdict(lambda: {"n": 0, "model": 0, "baseline": 0.0, "sure_n": 0, "sure_ok": 0})
for p in preds:
    q = key(p["case_id"])
    s = stats[q]
    s["n"] += 1
    ok = p["choice"] == p["expected"]
    s["model"] += ok
    if q in majority:
        s["baseline"] += p["expected"] == majority[q]
    else:  # price_field: a random pick among the price strings
        target = test_rows[p["case_id"]]["target"]
        s["baseline"] += sum(t > 0 for t in target) / len(target)
    if max(p["probabilities"].values()) >= 0.9:
        s["sure_n"] += 1
        s["sure_ok"] += ok

print(f"{'decision':12s} {'n':>5s} {'model':>7s} {'baseline':>9s}   {'confident>=0.9: coverage / accuracy':s}")
for q, s in sorted(stats.items()):
    base = "majority" if q in majority else "random"
    sure = f"{s['sure_n']/s['n']:.0%} / {s['sure_ok']/s['sure_n']:.1%}" if s["sure_n"] else "-"
    print(f"{q:12s} {s['n']:5d} {s['model']/s['n']:7.1%} {s['baseline']/s['n']:8.1%}({base})   {sure}")
RUN1 = {"page_type": 0.769, "price_field": 0.467, "in_stock": 0.882}
print("\nrun1 on its own test split:", RUN1, "(run2's test split differs: more product pages)")
print("\ncalibrated by type:", json.dumps(report["test_calibrated_by_type"]["all"], indent=1))
print("order-reversal flips:", report["order_reversal_argmax_change_fraction"])
print("latency:", report["latency"])

**Read the table like this:** the model must beat the baseline clearly. The most important
column is the last one: when the model is ≥0.9 sure, that accuracy should be 95%+. Those are the
pages it can handle on its own, and everything else goes to an LLM or a person.

### Julia-1 on the same test split

A general decision model of the same size class (SupersonicLabs/Julia-1, Apache-2.0). If Anarkali does
not clearly beat it on these web decisions, the specialist is not worth shipping.

In [ ]:
if JULIA_BASELINE:
    from huggingface_hub import snapshot_download
    julia_dir = snapshot_download("SupersonicLabs/Julia-1", local_dir=str(WORK / "Julia-1"))
    sh(f"pip install -q -e {julia_dir}")
    sys.path.insert(0, julia_dir)   # the julia/ package lives at the repo root
    from julia import load_model
    julia = load_model(julia_dir, device=DEVICE, strict_encoding=True, max_length=2048, head_length=512)

    def julia_question(row):
        kind = row.get("question_type", "choice")
        ids = [c["id"] for c in row["candidates"]]
        criteria = {c["id"]: c["text"] for c in row["candidates"]}
        text = row["question"].removeprefix("True or false: ")
        return {"type": kind, "instructions": text, "criteria": criteria}, ids

    julia_stats = defaultdict(lambda: [0, 0, 0])   # n, correct, skipped
    for row in test_rows.values():
        q = key(row["case_id"])
        spec, ids = julia_question(row)
        expected = ids[max(range(len(ids)), key=row["target"].__getitem__)]
        julia_stats[q][0] += 1
        if len(ids) > 20:
            julia_stats[q][2] += 1
            continue
        try:
            answer = julia.predict(state=json.dumps(row["state"], ensure_ascii=False), questions={"q": spec})["answers"]["q"]
        except Exception:
            julia_stats[q][2] += 1
            continue
        picked = answer.get("choice") if spec["type"] == "choice" else ("true" if answer["noul"] >= 0.5 else "false")
        julia_stats[q][1] += picked == expected
    print(f"{'decision':12s} {'Anarkali':>9s} {'Julia-1':>9s}  skipped")
    for q, (n, ok, skipped) in sorted(julia_stats.items()):
        print(f"{q:12s} {stats[q]['model']/stats[q]['n']:9.1%} {ok/n:9.1%}  {skipped}")
    julia_result = {q: v[1] / v[0] for q, v in julia_stats.items()}
else:
    julia_result = None

## 7. Try it on any HTML

In [ ]:
from anarkali.engine import Engine
from anarkali.web import page_state, PAGE_TYPE_QUESTION, IN_STOCK_QUESTION

engine = Engine.load(str(RUN / "best.pt"))
html = """<html><head><title>Steel Bottle 1L</title></head><body>
<h1>Steel Bottle 1L</h1><p>MRP <s>₹1,999</s> Deal price ₹1,299. Only 3 left in stock. Add to cart.</p>
<p>Free delivery. 7 day returns.</p></body></html>"""
state = page_state(html, "https://shop.example/p/steel-bottle")
price_options = {"a": "MRP [₹1,999] Deal price", "b": "Deal price [₹1,299]. Only 3 left"}
answer = engine.predict(state, {
    "page_type": PAGE_TYPE_QUESTION,
    "in_stock": IN_STOCK_QUESTION,
    "price_field": {"type": "choice", "instructions": "Which text shows this product's current selling price?",
                    "criteria": price_options},
})
print(json.dumps(answer["answers"], indent=1, ensure_ascii=False))

## 8. Export to ONNX (CPU inference, no torch)

In [ ]:
temps = json.dumps(report["temperature_by_type"])
sh(f"python scripts/export_onnx.py --checkpoint {RUN}/best.pt --output {RELEASE} --data {DATA} "
   f"--name 'Anarkali Web 150M' --temperature-by-type '{temps}' --no-int8")
shutil.copy(RUN / "test-report.json", RELEASE / "test-report.json")
print(sorted(p.name for p in RELEASE.iterdir()))

## 9. Save everything (best.pt included)

In [ ]:
OUT = WORK / "anarkali-web-output"
OUT.mkdir(exist_ok=True)
shutil.copy(RUN / "best.pt", OUT / "best.pt")
for name in ("training.json", "test-report.json", "anarkali-test.jsonl"):
    shutil.copy(RUN / name, OUT / name)
shutil.copy(DATA / "manifest.json", OUT / "dataset-manifest.json")
shutil.copy(LOG, OUT / "notebook-log.txt")
hf.upload_folder(repo_id=HF_REPO, folder_path=str(OUT), path_in_repo=f"{RUN_NAME}/web-run",
                 allow_patterns=["*.json", "*.jsonl", "*.txt"])
print("reports uploaded to", f"{HF_REPO}/{RUN_NAME}/web-run")
shutil.make_archive(str(WORK / "release-web"), "zip", RELEASE)
print("saved:", OUT, "and", WORK / "release-web.zip")

if not ON_KAGGLE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        target = Path("/content/drive/MyDrive/anarkali-web")
        shutil.copytree(OUT, target / "run", dirs_exist_ok=True)
        shutil.copy(WORK / "release-web.zip", target / "release-web.zip")
        print("copied to Google Drive:", target)
    except Exception as exc:
        print("Drive copy skipped:", exc)

In [ ]:
# Optional: the ONNX release too (CPU inference without torch).
hf.upload_folder(repo_id=HF_REPO, folder_path=str(RELEASE), path_in_repo=f"{RUN_NAME}/release-web")
print("ONNX release uploaded to", f"{HF_REPO}/{RUN_NAME}/release-web")